# Per-Label Analysis: NRPS, Polyketide, etc.
Focus: Individual label AUC-ROC and minor class performance

In [1]:
import pickle
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_auc_score, classification_report

# Load results
with open('../../results/mibig1_classification/complete_results.pkl', 'rb') as f:
    results = pickle.load(f)

print(f"Loaded {len(results)} models")
for r in results:
    print(f"- {r['model_name']}")
    
# Check structure
if results:
    sample = results[0]
    print(f"\nSample structure:")
    print(f"Keys: {list(sample.keys())}")
    if 'fold_results' in sample and sample['fold_results']:
        print(f"Fold keys: {list(sample['fold_results'][0].keys())}")
    if 'class_names' in sample:
        print(f"Classes: {sample['class_names']}")

Loaded 8 models
- ESM Init Last + BiLSTM
- ESM Init Embedder + BiLSTM
- Random Init Last + BiLSTM
- Random Init Embedder + BiLSTM
- ESM Embeddings + BiLSTM
- ESM + BigCarp Concatenated + BiLSTM
- Pfam2vec + Random Forest
- Random 256D Baseline

Sample structure:
Keys: ['model_name', 'embedding_column', 'fold_results', 'aggregate_metrics', 'class_names']
Fold keys: ['exact_match_accuracy', 'micro_f1', 'macro_f1', 'weighted_macro_f1', 'micro_auc', 'macro_auc', 'weighted_auc', 'fold']
Classes: ['Alkaloid', 'NRP', 'Other', 'Polyketide', 'RiPP', 'Saccharide', 'Terpene']


In [2]:
# Function to reconstruct per-class predictions from sklearn multilabel results
def reconstruct_per_class_data(results):
    per_class_data = []
    
    for result in results:
        model_name = result['model_name']
        
        # Try to get class names
        class_names = result.get('class_names', [])
        
        if not class_names:
            print(f"No class names for {model_name}")
            continue
            
        print(f"\nProcessing {model_name} with {len(class_names)} classes")
        
        # Collect predictions from all folds
        all_true = []
        all_proba = []
        
        for i, fold_result in enumerate(result['fold_results']):
            # Look for prediction data in fold
            found_data = False
            
            # Try different possible keys
            for true_key in ['y_true', 'y_test', 'true_labels', 'test_true']:
                for proba_key in ['y_proba', 'y_pred_proba', 'pred_proba', 'probabilities']:
                    if true_key in fold_result and proba_key in fold_result:
                        all_true.append(np.array(fold_result[true_key]))
                        all_proba.append(np.array(fold_result[proba_key]))
                        found_data = True
                        break
                if found_data:
                    break
            
            if not found_data:
                print(f"  Fold {i} keys: {list(fold_result.keys())}")
        
        if all_true and all_proba:
            # Concatenate all folds
            y_true_all = np.vstack(all_true)
            y_proba_all = np.vstack(all_proba)
            
            print(f"  Combined shape: {y_true_all.shape}")
            
            # Calculate per-class metrics
            for class_idx, class_name in enumerate(class_names):
                if class_idx < y_true_all.shape[1]:
                    class_true = y_true_all[:, class_idx]
                    class_proba = y_proba_all[:, class_idx]
                    
                    support = int(np.sum(class_true))
                    total = len(class_true)
                    frequency = support / total
                    
                    # Calculate AUC-ROC
                    auc_roc = np.nan
                    if len(np.unique(class_true)) > 1:  # Both classes present
                        try:
                            auc_roc = roc_auc_score(class_true, class_proba)
                        except Exception as e:
                            print(f"    AUC error for {class_name}: {e}")
                    
                    per_class_data.append({
                        'Model': model_name,
                        'Label': class_name,
                        'AUC_ROC': auc_roc,
                        'Support': support,
                        'Total': total,
                        'Frequency': frequency
                    })
        else:
            print(f"  No prediction data found for {model_name}")
    
    return pd.DataFrame(per_class_data)

# Extract per-class data
per_class_df = reconstruct_per_class_data(results)
print(f"\nFinal per-class dataframe: {per_class_df.shape}")
if not per_class_df.empty:
    per_class_df.head(10)


Processing ESM Init Last + BiLSTM with 7 classes
  Fold 0 keys: ['exact_match_accuracy', 'micro_f1', 'macro_f1', 'weighted_macro_f1', 'micro_auc', 'macro_auc', 'weighted_auc', 'fold']
  Fold 1 keys: ['exact_match_accuracy', 'micro_f1', 'macro_f1', 'weighted_macro_f1', 'micro_auc', 'macro_auc', 'weighted_auc', 'fold']
  Fold 2 keys: ['exact_match_accuracy', 'micro_f1', 'macro_f1', 'weighted_macro_f1', 'micro_auc', 'macro_auc', 'weighted_auc', 'fold']
  Fold 3 keys: ['exact_match_accuracy', 'micro_f1', 'macro_f1', 'weighted_macro_f1', 'micro_auc', 'macro_auc', 'weighted_auc', 'fold']
  Fold 4 keys: ['exact_match_accuracy', 'micro_f1', 'macro_f1', 'weighted_macro_f1', 'micro_auc', 'macro_auc', 'weighted_auc', 'fold']
  No prediction data found for ESM Init Last + BiLSTM

Processing ESM Init Embedder + BiLSTM with 7 classes
  Fold 0 keys: ['exact_match_accuracy', 'micro_f1', 'macro_f1', 'weighted_macro_f1', 'micro_auc', 'macro_auc', 'weighted_auc', 'fold']
  Fold 1 keys: ['exact_match_acc

In [3]:
# If direct extraction fails, use aggregate metrics approach
if per_class_df.empty:
    print("Direct extraction failed. Using aggregate metrics...")
    
    # At least show model comparison
    summary_data = []
    for result in results:
        metrics = result['aggregate_metrics']
        summary_data.append({
            'Model': result['model_name'],
            'Macro_AUC': metrics.get('macro_auc', np.nan),
            'Weighted_AUC': metrics.get('weighted_auc', np.nan),
            'Macro_F1': metrics.get('macro_f1', np.nan)
        })
    
    summary_df = pd.DataFrame(summary_data)
    print("\nModel-level performance:")
    print(summary_df.round(4))
else:
    # Analyze label distribution
    label_stats = per_class_df.groupby('Label').agg({
        'Support': 'first',
        'Total': 'first', 
        'Frequency': 'first',
        'AUC_ROC': 'mean'
    }).sort_values('Frequency')
    
    print("\n📊 Label Distribution (sorted by frequency):")
    print(label_stats.round(4))

Direct extraction failed. Using aggregate metrics...

Model-level performance:
                                 Model  Macro_AUC  Weighted_AUC  Macro_F1
0               ESM Init Last + BiLSTM     0.9128        0.9388    0.7027
1           ESM Init Embedder + BiLSTM     0.8873        0.9167    0.6555
2            Random Init Last + BiLSTM     0.9233        0.9424    0.7096
3        Random Init Embedder + BiLSTM     0.9196        0.9412    0.7219
4              ESM Embeddings + BiLSTM     0.8955        0.9224    0.6879
5  ESM + BigCarp Concatenated + BiLSTM     0.9075        0.9370    0.7180
6             Pfam2vec + Random Forest     0.9140        0.9338    0.5933
7                 Random 256D Baseline     0.4934        0.4843    0.1672


In [4]:
if not per_class_df.empty:
    # Define minor vs major classes
    minor_threshold = 0.05  # < 5% frequency
    
    minor_labels = label_stats[label_stats['Frequency'] < minor_threshold].index.tolist()
    major_labels = label_stats[label_stats['Frequency'] >= minor_threshold].index.tolist()
    
    print(f"Minor labels ({len(minor_labels)}): {minor_labels}")
    print(f"Major labels ({len(major_labels)}): {major_labels}")
    
    # Add label type column
    per_class_df['Label_Type'] = per_class_df['Label'].apply(
        lambda x: 'Minor' if x in minor_labels else 'Major'
    )
    
    # Performance by label type
    type_performance = per_class_df.groupby(['Model', 'Label_Type'])['AUC_ROC'].agg([
        'mean', 'std', 'count', 'min', 'max'
    ]).round(4)
    
    print("\n🎯 Performance by Label Type:")
    print(type_performance)

In [5]:
if not per_class_df.empty:
    # Best models for minor classes
    minor_performance = per_class_df[per_class_df['Label_Type'] == 'Minor'].groupby('Model')['AUC_ROC'].mean().sort_values(ascending=False)
    
    print("🏆 Best models for minor classes (by mean AUC-ROC):")
    for model, score in minor_performance.items():
        print(f"{model:40s}: {score:.4f}")
    
    # Worst performing labels across all models
    worst_labels = per_class_df.groupby('Label')['AUC_ROC'].mean().sort_values().head(10)
    print("\n⚠️ Worst performing labels (bottom 10):")
    for label, score in worst_labels.items():
        support = label_stats.loc[label, 'Support']
        freq = label_stats.loc[label, 'Frequency']
        print(f"{label:20s}: AUC={score:.3f} (n={support:3d}, {freq*100:.1f}%)")

In [6]:
if not per_class_df.empty:
    # Heatmap: AUC-ROC by model and label
    pivot_df = per_class_df.pivot(index='Label', columns='Model', values='AUC_ROC')
    pivot_df = pivot_df.loc[label_stats.index]  # Order by frequency
    
    plt.figure(figsize=(14, max(8, len(pivot_df.index) * 0.3)))
    sns.heatmap(pivot_df, annot=True, fmt='.3f', cmap='RdYlBu_r',
                vmin=0.5, vmax=1.0, cbar_kws={'label': 'AUC-ROC'})
    plt.title('Per-Label AUC-ROC by Model\n(Labels ordered by frequency: rare to common)')
    plt.xlabel('Model')
    plt.ylabel('Label')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

In [7]:
if not per_class_df.empty:
    # Box plot: Minor vs Major label performance
    plt.figure(figsize=(12, 6))
    sns.boxplot(data=per_class_df, x='Model', y='AUC_ROC', hue='Label_Type')
    plt.title('AUC-ROC Distribution: Minor vs Major Labels')
    plt.xticks(rotation=45, ha='right')
    plt.axhline(y=0.5, color='red', linestyle='--', alpha=0.5, label='Random')
    plt.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
if not per_class_df.empty:
    # Save detailed results
    import os
    os.makedirs('../../results/mibig1_classification', exist_ok=True)
    
    per_class_df.to_csv('../../results/mibig1_classification/per_label_aucroc.csv', index=False)
    label_stats.to_csv('../../results/mibig1_classification/label_distribution.csv')
    type_performance.to_csv('../../results/mibig1_classification/label_type_performance.csv')
    
    print("💾 Results saved:")
    print("- per_label_aucroc.csv")
    print("- label_distribution.csv") 
    print("- label_type_performance.csv")
else:
    print("⚠️ No per-label data to save. Check if results contain prediction arrays.")